# Organização do ambiente
Cria catálogo, schemas e os volumes externos já usados pela carga. A credencial IAM e a localização externa S3 devem existir e permitir as operações abaixo. Não remove objetos ou dados. Preencha DESTINATARIO somente para conceder leitura; a identidade executora precisa das permissões de criação/escrita.

In [ ]:
CATALOGO = "dengue_sus"
BASE_S3 = "s3://pcd2-dengue/pysus-ibge/landing"
DESTINATARIO = ""  # Usuário ou grupo; vazio não altera concessões.

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for camada in ("landing", "bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{camada}")
for volume, pasta in (("inmet", "inmet"), ("sinan_s3", "sinan")):
    nome, origem = f"{CATALOGO}.landing.{volume}", f"{BASE_S3}/{pasta}/"
    spark.sql(f"CREATE EXTERNAL VOLUME IF NOT EXISTS {nome} LOCATION '{origem}'")
    salvo = spark.sql(f"DESCRIBE VOLUME {nome}").first()
    if salvo["volume_type"] != "EXTERNAL" or salvo["storage_location"].rstrip("/") != origem.rstrip("/"):
        raise ValueError(f"{nome}: tipo/localização incompatíveis; não houve substituição.")

if DESTINATARIO:
    destinatario = DESTINATARIO.replace("`", "``")
    spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `{destinatario}`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.silver TO `{destinatario}`")
    spark.sql(f"GRANT SELECT ON SCHEMA {CATALOGO}.silver TO `{destinatario}`")
print("Ambiente configurado. Execute as validações Landing e depois as cargas Bronze/Silver.")